In [ ]:
import random
import string
from datasets import load_dataset, get_dataset_config_names

# =================================================================
# ⭐ 데이터셋 설명 및 목표: ExamBench (시험 대비 코퍼스)
#
# 이 데이터셋은 JEE, NEET, UPSC 등 전 세계의 경쟁 시험을 위한
# 방대한 양의 문제 풀이와 사고 과정(Chain-of-Thought)을 담고 있습니다.
# 🧠 학습 목표: 이 데이터셋을 분석하여, 단순히 정답을 찾는 것(Question Answering)
# 을 넘어, 왜 이 정답에 도달했는지 단계별 추론 과정(Complex CoT)을 분석하고
# 구조화하는 능력을 기르는 방법을 코드로 실습합니다.
#
# 사용 기능: prompt (문제), complex_cot (풀이 과정), response (정답)
# =================================================================

# --- 환경 설정 상수 ---
DATASET_NAME = "169Pi/exambench"
SAMPLE_COUNT = 5  # ✨ 초보자를 위해 상위 5개 샘플만 분석합니다.
# =================================================================

print("💡 튜터링 시작! AI의 두뇌 구조 분석 실습에 오신 것을 환영합니다!")

# 1. 사용 가능한 Config 목록 확인 (필수 패턴 준수)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"\n✅ 사용 가능한 Config 목록: {configs}")
    # 이 데이터셋은 기본 설정만 사용합니다.
    selected_config = None 
except Exception as e:
    print(f"\nℹ️ Config 확인 중 오류가 발생했거나 기본 설정만 사용합니다. ({e})")
    selected_config = None

# 2. 데이터 로드 (스트리밍 모드 우선 시도)
dataset = None
try:
    # ✅ 1단계: 스트리밍 방식으로 데이터셋을 로드합니다. (대용량 데이터 처리에 최적)
    print("\n⏳ 1. 스트리밍 모드 (streaming=True)로 데이터셋 로드를 시도합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("🌟 스트리밍 모드 로드 성공! 대용량 처리가 준비되었습니다.")

except Exception as e:
    # ⚠️ 만약 스트리밍 로드에 실패할 경우 (예: 환경 문제), 일반 모드로 다운그레이드
    print(f"\n🚨 스트리밍 로드에 실패했습니다. ({e}). 일반 다운로드 모드로 전환합니다...")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("🌟 일반 다운로드 모드 로드 성공! 데이터셋을 메모리에 모두 가져왔습니다.")
    except Exception as e_fallback:
        print(f"❌ 죄송해요. 데이터셋 로드에 최종적으로 실패했습니다: {e_fallback}")
        exit()

# 3. 샘플링 및 반복 처리 준비
if dataset:
    print(f"\n✨ 2. 총 {DATASET_NAME} 데이터셋에서 {SAMPLE_COUNT}개의 샘플을 추출합니다.")
    
    # Constraint 9: .take()를 사용하여 상위 K개만 가져옵니다.
    sampled_dataset = dataset.take(SAMPLE_COUNT)
    
    # Constraint 9: 스트리밍 모드에 맞춰 iterator 패턴을 사용합니다.
    sample_iterator = iter(sampled_dataset)

    # 3.1. 데이터 분석 및 구조화 함수 (Creative Use Case 구현)
    def analyze_sample(sample):
        """
        단일 샘플을 분석하여 '문제 유형'과 '추론 난이도' 등의 메트릭을 추정합니다.
        """
        prompt = sample.get('prompt', '')
        cot = sample.get('complex_cot', '')
        response = sample.get('response', '')

        # 문제의 길이를 측정하여 난이도 지표로 활용 (단어 수 기반)
        prompt_length = len(prompt.split())
        
        # 풀이 과정의 길이를 측정 (CoT가 길수록 복잡한 추론이 필요하다고 추정)
        cot_length = len(cot.split())
        
        # 최종 점수 (가상) 계산: 문제 난이도 * CoT 길이
        difficulty_score = prompt_length + (cot_length // 2)
        
        print("-" * 50)
        print(f"  [📝 샘플 분석 결과] 난이도 점수: {difficulty_score}점")
        print(f"  [💡 추정 과정] 문제 길이({prompt_length} 단어)와 추론 과정({cot_length} 단어)을 분석했습니다.")
        print(f"  [🎯 문제] {prompt[:70]}...")
        print(f"  [🧠 추론 과정] {cot[:70]}...")
        print(f"  [✅ 정답] {response[:30]}...")
        print("-" * 50)
        
        return {
            "prompt_len": prompt_length,
            "cot_len": cot_length,
            "score": difficulty_score
        }

    # 3.2. 반복문으로 샘플을 순회하며 실습 수행
    sample_count = 0
    sample_data_list = []
    print("\n🚀 3. 본격적인 샘플 분석 루프 시작!")
    
    while sample_count < SAMPLE_COUNT:
        try:
            # Constraint 16: next() 함수를 사용하여 다음 샘플을 강제로 조회합니다.
            sample_data = next(sample_iterator)
            
            # 데이터 분석 함수 호출
            analysis = analyze_sample(sample_data)
            sample_data_list.append(analysis)
            sample_count += 1

        except StopIteration:
            # 더 이상 샘플이 없을 때 종료
            break

    # 4. 최종 결과 요약 (정량적 분석)
    print("\n=============================================================")
    print("📊 4. 전체 샘플에 대한 정량적 분석 및 요약 (튜터 피드백)")
    print("=============================================================")
    
    if sample_data_list:
        total_score = sum(d['score'] for d in sample_data_list)
        avg_score = total_score / len(sample_data_list)
        
        print(f"✅ 총 분석한 샘플 수: {len(sample_data_list)}개")
        print(f"📈 평균 난이도 점수: {avg_score:.2f}점")
        print(f"✨ 분석을 통해, 단순 답변(Response)보다 '사고 과정(Complex CoT)'을 분석하는 것이")
        print("   AI가 가장 어려운 능력을 배우는 핵심 지점입니다! 😉")
    else:
        print("😢 분석할 샘플 데이터를 찾을 수 없었습니다.")

print("\n🎉 실습 완료! 복잡한 텍스트를 구조적으로 분석하는 능력을 잘 보여주었어요!")